**RoBERTa DATA LOADING + CLEANING**

In [ ]:
# ============================================================
# CELL 1
# FINAL ROBUST DATA LOADING FOR COLAB
# FIXES ALL CSV PARSER ISSUES
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import torch
import warnings

from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# ============================================================
# DEVICE
# ============================================================

print("="*100)
print("DEVICE")
print("="*100)

device = torch.device(

    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"\nUsing Device: {device}")

# ============================================================
# DATA PATH
# ============================================================

from pathlib import Path

# Kaggle input path is discovered from the dataset attached in the notebook sidebar.
_dataset_files = sorted(Path("/kaggle/input").rglob("bias_clean.csv"))
if not _dataset_files:
    raise FileNotFoundError("Attach the bias_clean Kaggle dataset (ponssvvv/bias-clean) in the notebook Input panel.")
DATA_PATH = str(_dataset_files[0])
print(f"Using dataset: {DATA_PATH}")

# ============================================================
# LOAD DATASET
# ============================================================

print("\n" + "="*100)
print("LOADING DATASET")
print("="*100)

df = pd.read_csv(

    DATA_PATH,

    sep=",",

    engine="python",

    encoding="utf-8",

    on_bad_lines="skip"
)

print(f"\nDataset loaded from:\n{DATA_PATH}")

# ============================================================
# DATASET INFO
# ============================================================

print("\nDataset Columns:\n")

print(df.columns)

# ============================================================
# TEXT COLUMN
# ============================================================

TEXT_COLUMN = "page_text"

print(f"\nUsing Text Column: {TEXT_COLUMN}")

# ============================================================
# KEEP REQUIRED COLUMNS
# ============================================================

df = df[[TEXT_COLUMN, "bias"]]

# ============================================================
# REMOVE NULLS
# ============================================================

df = df.dropna()

# ============================================================
# REMOVE EMPTY TEXT
# ============================================================

df = df[

    df[TEXT_COLUMN].astype(str).str.strip() != ""
]

# ============================================================
# SIMPLE TEXT CLEANING
# ============================================================

df[TEXT_COLUMN] = (

    df[TEXT_COLUMN]

    .astype(str)

    .str.replace("\n", " ", regex=False)

    .str.replace("\r", " ", regex=False)

    .str.replace("\t", " ", regex=False)

    .str.strip()
)

# ============================================================
# LABEL MAPPING
# ============================================================

label_mapping = {

    "center": 0,

    "leaning-left": 1,

    "leaning-right": 2,

    "left": 3,

    "right": 4
}

# ============================================================
# APPLY LABELS
# ============================================================

df["label"] = df["bias"].map(label_mapping)

# ============================================================
# REMOVE INVALID LABELS
# ============================================================

df = df.dropna(subset=["label"])

df["label"] = df["label"].astype(int)

# ============================================================
# FINAL DATASET SIZE
# ============================================================

print(f"\nClean dataset size: {len(df)}")

# ============================================================
# LABEL MAPPING DISPLAY
# ============================================================

print("\nLabel Mapping:\n")

for key, value in label_mapping.items():

    print(f"{key} → {value}")

# ============================================================
# ORIGINAL DISTRIBUTION
# ============================================================

print("\nOriginal Class Distribution:\n")

print(df["bias"].value_counts())

# ============================================================
# TRAIN / VALID / TEST SPLIT
# ============================================================

train_df, temp_df = train_test_split(

    df,

    test_size=0.30,

    stratify=df["label"],

    random_state=42
)

valid_df, test_df = train_test_split(

    temp_df,

    test_size=0.50,

    stratify=temp_df["label"],

    random_state=42
)

# ============================================================
# RESET INDEX
# ============================================================

train_df = train_df.reset_index(drop=True)

valid_df = valid_df.reset_index(drop=True)

test_df = test_df.reset_index(drop=True)

# ============================================================
# SPLIT INFO
# ============================================================

print("\nDataset Split:\n")

print(f"Train Size      : {len(train_df)}")

print(f"Validation Size : {len(valid_df)}")

print(f"Test Size       : {len(test_df)}")

# ============================================================
# TRAIN DISTRIBUTION
# ============================================================

reverse_mapping = {

    v:k for k,v in label_mapping.items()
}

print("\nTrain Class Distribution:\n")

train_counts = train_df["label"].value_counts().sort_index()

for idx, count in train_counts.items():

    print(f"{reverse_mapping[idx]:15} : {count}")

# ============================================================
# SUCCESS
# ============================================================

print("\n" + "="*100)
print("CELL 1 COMPLETED SUCCESSFULLY")
print("="*100)

**CELL 2 - TOKENIZATION + WEIGHTED BALANCED SAMPLER**

In [ ]:
# ============================================================
# CELL 2
# FINAL CORRECTED ROBERTA TRAINING PIPELINE
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (

    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    cohen_kappa_score,
    confusion_matrix,
    classification_report
)

from datasets import Dataset

from transformers import (

    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from torch.nn import CrossEntropyLoss

# ============================================================
# REMOVE OLD CHECKPOINTS
# ============================================================

if os.path.exists("./roberta_results"):

    shutil.rmtree("./roberta_results")

print("="*100)
print("OLD CHECKPOINTS REMOVED")
print("="*100)

# ============================================================
# REVERSE LABEL MAPPING
# ============================================================

reverse_mapping = {

    v:k for k,v in label_mapping.items()
}

# ============================================================
# MODEL CONFIG
# ============================================================

MODEL_NAME = "roberta-base"

MAX_LENGTH = 256

# ============================================================
# LOAD TOKENIZER
# ============================================================

print("\n" + "="*100)
print("LOADING ROBERTA TOKENIZER")
print("="*100)

tokenizer = AutoTokenizer.from_pretrained(

    MODEL_NAME
)

print("\nTokenizer Loaded Successfully.")

print(f"\nUsing MAX_LENGTH = {MAX_LENGTH}")

# ============================================================
# TOKENIZATION FUNCTION
# ============================================================

def tokenize_function(examples):

    return tokenizer(

        examples["page_text"],

        truncation=True,

        padding="max_length",

        max_length=MAX_LENGTH
    )

# ============================================================
# TRAINING DISTRIBUTION
# ============================================================

print("\n" + "="*100)
print("TRAINING CLASS DISTRIBUTION")
print("="*100)

train_distribution = train_df["label"].value_counts().sort_index()

for idx, count in train_distribution.items():

    print(f"{reverse_mapping[idx]:15} : {count}")

# ============================================================
# CLASS WEIGHTS
# ============================================================

class_weights = compute_class_weight(

    class_weight="balanced",

    classes=np.unique(train_df["label"]),

    y=train_df["label"]
)

class_weights = torch.tensor(

    class_weights,

    dtype=torch.float
).to(device)

print("\nClass Weights:\n")

for idx, weight in enumerate(class_weights):

    print(f"{reverse_mapping[idx]:15} : {weight:.4f}")

# ============================================================
# EFFECTIVE BALANCED DISTRIBUTION
# ============================================================

print("\n" + "="*100)
print("EFFECTIVE BALANCED TRAINING DISTRIBUTION")
print("="*100)

effective_weights = {}

for idx, count in train_distribution.items():

    effective_value = count * class_weights[idx].item()

    effective_weights[

        reverse_mapping[idx]
    ] = int(effective_value)

    print(

        f"{reverse_mapping[idx]:15} : {int(effective_value)}"
    )

# ============================================================
# BALANCING PLOT
# ============================================================

original_counts = [

    train_distribution[idx]

    for idx in sorted(train_distribution.index)
]

effective_counts = [

    effective_weights[reverse_mapping[idx]]

    for idx in sorted(train_distribution.index)
]

class_names = [

    reverse_mapping[idx]

    for idx in sorted(train_distribution.index)
]

x = np.arange(len(class_names))

width = 0.35

plt.figure(figsize=(12,6))

plt.bar(

    x - width/2,
    original_counts,
    width,
    label='Original'
)

plt.bar(

    x + width/2,
    effective_counts,
    width,
    label='Weighted Effective'
)

plt.xticks(

    x,
    class_names,
    rotation=20
)

plt.ylabel("Count")

plt.title("Original vs Effective Weighted Distribution")

plt.legend()

plt.grid(axis='y')

plt.show()

# ============================================================
# HF DATASETS
# ============================================================

train_dataset = Dataset.from_pandas(

    train_df[["page_text", "label"]]
)

valid_dataset = Dataset.from_pandas(

    valid_df[["page_text", "label"]]
)

test_dataset = Dataset.from_pandas(

    test_df[["page_text", "label"]]
)

# ============================================================
# TOKENIZATION
# ============================================================

train_dataset = train_dataset.map(

    tokenize_function,
    batched=True
)

valid_dataset = valid_dataset.map(

    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(

    tokenize_function,
    batched=True
)

# ============================================================
# FORMAT DATASETS
# ============================================================

train_dataset.set_format(

    type="torch",

    columns=[

        "input_ids",
        "attention_mask",
        "label"
    ]
)

valid_dataset.set_format(

    type="torch",

    columns=[

        "input_ids",
        "attention_mask",
        "label"
    ]
)

test_dataset.set_format(

    type="torch",

    columns=[

        "input_ids",
        "attention_mask",
        "label"
    ]
)

# ============================================================
# LOAD ROBERTA MODEL
# ============================================================

print("\n" + "="*100)
print("LOADING ROBERTA MODEL")
print("="*100)

model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=len(label_mapping),

    hidden_dropout_prob=0.3,

    attention_probs_dropout_prob=0.3
)

model.to(device)

print("\nModel Loaded Successfully.")

# ============================================================
# CUSTOM WEIGHTED TRAINER
# ============================================================

class WeightedTrainer(Trainer):

    def compute_loss(

        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs
    ):

        labels = inputs.get("labels")

        outputs = model(

            input_ids=inputs.get("input_ids"),

            attention_mask=inputs.get("attention_mask")
        )

        logits = outputs.get("logits")

        num_labels = (

            model.module.config.num_labels

            if hasattr(model, "module")

            else model.config.num_labels
        )

        loss_fct = CrossEntropyLoss(

            weight=class_weights
        )

        loss = loss_fct(

            logits.view(-1, num_labels),

            labels.view(-1)
        )

        return (

            (loss, outputs)

            if return_outputs

            else loss
        )

# ============================================================
# METRICS FUNCTION
# ============================================================

def compute_metrics(pred):

    labels = pred.label_ids

    preds = pred.predictions.argmax(-1)

    accuracy = accuracy_score(labels, preds)

    precision = precision_score(

        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(

        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(

        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    return {

        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# ============================================================
# TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(

    output_dir="./roberta_results",

    num_train_epochs=40,

    learning_rate=1e-5,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    gradient_accumulation_steps=2,

    warmup_ratio=0.1,

    weight_decay=0.05,

    max_grad_norm=1.0,

    logging_steps=100,

    eval_strategy="epoch",

    save_strategy="epoch",

    save_total_limit=1,

    save_only_model=True,

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    do_train=True,

    do_eval=True,

    fp16=False,

    seed=42
)

# ============================================================
# TRAINER
# ============================================================

trainer = WeightedTrainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    compute_metrics=compute_metrics,

    callbacks=[

        EarlyStoppingCallback(

            early_stopping_patience=4
        )
    ]
)

# ============================================================
# START TRAINING
# ============================================================

print("\n" + "="*100)
print("STARTING ROBERTA TRAINING")
print("="*100)

trainer.train()

In [ ]:
# ============================================================
# SINGLE IMPROVED EVALUATION CELL
# ROBERTA ADVANCED EVALUATION + ALL CURVES
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.metrics import (

    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_curve,
    auc,
    precision_recall_curve
)

from sklearn.preprocessing import label_binarize

# ============================================================
# REVERSE LABEL MAPPING
# ============================================================

reverse_mapping = {

    v:k for k,v in label_mapping.items()
}

# ============================================================
# GENERATE TEST PREDICTIONS
# ============================================================

print("="*100)
print("GENERATING TEST PREDICTIONS")
print("="*100)

predictions = trainer.predict(

    test_dataset
)

preds = np.argmax(

    predictions.predictions,

    axis=1
)

labels = predictions.label_ids

print("\nPredictions Generated Successfully.")

# ============================================================
# PROBABILITIES
# ============================================================

print("\n" + "="*100)
print("GENERATING PROBABILITIES")
print("="*100)

probabilities = torch.softmax(

    torch.tensor(predictions.predictions),

    dim=1
).numpy()

# ============================================================
# ADVANCED METRICS
# ============================================================

accuracy = accuracy_score(

    labels,
    preds
)

precision = precision_score(

    labels,
    preds,
    average="weighted",
    zero_division=0
)

recall = recall_score(

    labels,
    preds,
    average="weighted",
    zero_division=0
)

weighted_f1 = f1_score(

    labels,
    preds,
    average="weighted",
    zero_division=0
)

macro_f1 = f1_score(

    labels,
    preds,
    average="macro",
    zero_division=0
)

mcc = matthews_corrcoef(

    labels,
    preds
)

kappa = cohen_kappa_score(

    labels,
    preds
)

# ============================================================
# METRICS DISPLAY
# ============================================================

print("\n" + "="*100)
print("ADVANCED EVALUATION METRICS")
print("="*100)

print(f"\nAccuracy           : {accuracy:.4f}")

print(f"Precision          : {precision:.4f}")

print(f"Recall             : {recall:.4f}")

print(f"Weighted F1        : {weighted_f1:.4f}")

print(f"Macro F1           : {macro_f1:.4f}")

print(f"Matthews Corrcoef  : {mcc:.4f}")

print(f"Cohen Kappa        : {kappa:.4f}")

# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\n" + "="*100)
print("CLASSIFICATION REPORT")
print("="*100)

print(

    classification_report(

        labels,
        preds,

        target_names=[

            reverse_mapping[i]

            for i in sorted(reverse_mapping.keys())
        ],

        zero_division=0
    )
)

# ============================================================
# CONFUSION MATRIX
# ============================================================

print("\n" + "="*100)
print("CONFUSION MATRIX")
print("="*100)

cm = confusion_matrix(

    labels,
    preds
)

plt.figure(figsize=(10,8))

sns.heatmap(

    cm,

    annot=True,

    fmt="d",

    cmap="Blues",

    xticklabels=[

        reverse_mapping[i]

        for i in sorted(reverse_mapping.keys())
    ],

    yticklabels=[

        reverse_mapping[i]

        for i in sorted(reverse_mapping.keys())
    ]
)

plt.title("RoBERTa Confusion Matrix")

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()

# ============================================================
# ROC CURVES
# ============================================================

print("\n" + "="*100)
print("ROC CURVES")
print("="*100)

y_true_bin = label_binarize(

    labels,

    classes=np.unique(labels)
)

n_classes = y_true_bin.shape[1]

plt.figure(figsize=(10,8))

for i in range(n_classes):

    fpr, tpr, _ = roc_curve(

        y_true_bin[:, i],

        probabilities[:, i]
    )

    roc_auc = auc(

        fpr,
        tpr
    )

    plt.plot(

        fpr,
        tpr,

        linewidth=2,

        label=f"{reverse_mapping[i]} (AUC={roc_auc:.2f})"
    )

plt.plot(

    [0,1],
    [0,1],

    linestyle="--"
)

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curves")

plt.legend()

plt.grid(True)

plt.show()

# ============================================================
# PRECISION-RECALL CURVES
# ============================================================

print("\n" + "="*100)
print("PRECISION-RECALL CURVES")
print("="*100)

plt.figure(figsize=(10,8))

for i in range(n_classes):

    precision_curve, recall_curve, _ = precision_recall_curve(

        y_true_bin[:, i],

        probabilities[:, i]
    )

    plt.plot(

        recall_curve,
        precision_curve,

        linewidth=2,

        label=f"{reverse_mapping[i]}"
    )

plt.xlabel("Recall")

plt.ylabel("Precision")

plt.title("Precision-Recall Curves")

plt.legend()

plt.grid(True)

plt.show()

# ============================================================
# CONFIDENCE DISTRIBUTION
# ============================================================

print("\n" + "="*100)
print("CONFIDENCE DISTRIBUTION")
print("="*100)

confidence_scores = np.max(

    probabilities,

    axis=1
)

plt.figure(figsize=(10,6))

sns.histplot(

    confidence_scores,

    bins=30
)

plt.title("Prediction Confidence Distribution")

plt.xlabel("Confidence Score")

plt.ylabel("Frequency")

plt.show()

# ============================================================
# PREDICTION DISTRIBUTION
# ============================================================

print("\n" + "="*100)
print("PREDICTION DISTRIBUTION")
print("="*100)

predicted_names = [

    reverse_mapping[p]

    for p in preds
]

plt.figure(figsize=(10,6))

sns.countplot(

    x=predicted_names
)

plt.title("Prediction Distribution")

plt.xticks(rotation=20)

plt.show()

# ============================================================
# PER-CLASS ACCURACY
# ============================================================

print("\n" + "="*100)
print("PER-CLASS ACCURACY")
print("="*100)

per_class_accuracy = {}

for class_id in np.unique(labels):

    idxs = np.where(

        np.array(labels) == class_id
    )[0]

    acc = accuracy_score(

        np.array(labels)[idxs],

        np.array(preds)[idxs]
    )

    per_class_accuracy[

        reverse_mapping[class_id]
    ] = acc

plt.figure(figsize=(10,6))

plt.bar(

    per_class_accuracy.keys(),

    per_class_accuracy.values()
)

plt.title("Per-Class Accuracy")

plt.ylabel("Accuracy")

plt.ylim(0,1)

plt.xticks(rotation=20)

plt.grid(axis='y')

plt.show()

# ============================================================
# LOSS CURVES
# ============================================================

print("\n" + "="*100)
print("LOSS CURVES")
print("="*100)

logs = trainer.state.log_history

train_loss = []

eval_loss = []

train_epochs = []

eval_epochs = []

for log in logs:

    if "loss" in log and "epoch" in log:

        train_loss.append(log["loss"])

        train_epochs.append(log["epoch"])

    if "eval_loss" in log:

        eval_loss.append(log["eval_loss"])

        eval_epochs.append(log["epoch"])

# ============================================================
# TRAINING LOSS
# ============================================================

plt.figure(figsize=(10,6))

plt.plot(

    train_epochs,
    train_loss,

    marker='o',

    linewidth=2
)

plt.xlabel("Epoch")

plt.ylabel("Training Loss")

plt.title("Training Loss Curve")

plt.grid(True)

plt.show()

# ============================================================
# VALIDATION LOSS
# ============================================================

plt.figure(figsize=(10,6))

plt.plot(

    eval_epochs,
    eval_loss,

    marker='o',

    linewidth=2,
    color='orange'
)

plt.xlabel("Epoch")

plt.ylabel("Validation Loss")

plt.title("Validation Loss Curve")

plt.grid(True)

plt.show()



# ============================================================
# CONFIDENCE vs CORRECTNESS
# ============================================================

print("\n" + "="*100)
print("CONFIDENCE vs CORRECTNESS")
print("="*100)

correctness = (

    np.array(preds) == np.array(labels)
).astype(int)

plt.figure(figsize=(10,6))

plt.scatter(

    confidence_scores,
    correctness,

    alpha=0.5
)

plt.xlabel("Confidence Score")

plt.ylabel("Correct Prediction")

plt.title("Confidence vs Correctness")

plt.grid(True)

plt.show()

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "="*100)
print("FINAL MODEL SUMMARY")
print("="*100)

print(f"""

FINAL PERFORMANCE

Accuracy        : {accuracy:.4f}
Weighted F1     : {weighted_f1:.4f}
Macro F1        : {macro_f1:.4f}
MCC             : {mcc:.4f}
Cohen Kappa     : {kappa:.4f}

MODEL

RoBERTa-base
5-Class Political Bias Classification

FEATURES USED

✓ Weighted Loss
✓ Early Stopping
✓ F1-Based Training
✓ Class Balancing
✓ Advanced Metrics
✓ Full Curve Analysis

VISUALIZATIONS

✓ Confusion Matrix
✓ ROC Curves
✓ Precision-Recall Curves
✓ Confidence Distribution
✓ Prediction Distribution
✓ Per-Class Accuracy
✓ Training Loss Curve
✓ Validation Loss Curve
✓ Overfitting Analysis
✓ Confidence vs Correctness

""")

print("="*100)
print("EVALUATION COMPLETED SUCCESSFULLY")
print("="*100)

In [ ]:
# Reproducibility export: metrics and evaluation artifacts for the Kaggle output bundle.
from pathlib import Path
import json
import pandas as pd

_output_dir = Path("/kaggle/working/results")
_output_dir.mkdir(parents=True, exist_ok=True)
_metrics = {
    "model": 'RoBERTa',
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "weighted_f1": float(weighted_f1),
    "macro_f1": float(macro_f1),
    "matthews_corrcoef": float(mcc),
    "cohen_kappa": float(kappa),
    "test_samples": int(len(test_df)),
}
(_output_dir / "roberta_metrics.json").write_text(json.dumps(_metrics, indent=2), encoding="utf-8")
if "cm" in globals():
    pd.DataFrame(cm).to_csv(_output_dir / "roberta_confusion_matrix.csv", index=False)
if "labels" in globals() and "preds" in globals():
    from sklearn.metrics import classification_report
    _names = list(label_mapping.values()) if "label_mapping" in globals() else None
    _report = classification_report(labels, preds, target_names=_names, output_dict=True, zero_division=0)
    pd.DataFrame(_report).transpose().to_csv(_output_dir / "roberta_classification_report.csv")
print("Saved metrics and evaluation artifacts under /kaggle/working/results")

In [ ]:
# ============================================================
# EXTRA RESEARCH-GRADE ANALYSIS
# PARAMETER SUMMARY + SAMPLE PREDICTIONS + LR CURVE
# ============================================================

# ============================================================
# MODEL PARAMETER SUMMARY
# ============================================================

print("="*100)
print("MODEL PARAMETER SUMMARY")
print("="*100)

total_params = sum(

    p.numel()

    for p in model.parameters()
)

trainable_params = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad
)

non_trainable_params = (

    total_params - trainable_params
)

print(f"\nTotal Parameters         : {total_params:,}")

print(f"Trainable Parameters     : {trainable_params:,}")

print(f"Non-Trainable Parameters : {non_trainable_params:,}")

# ============================================================
# LEARNING RATE CURVE
# ============================================================

print("\n" + "="*100)
print("LEARNING RATE CURVE")
print("="*100)

logs = trainer.state.log_history

learning_rates = []

lr_epochs = []

for log in logs:

    if "learning_rate" in log and "epoch" in log:

        learning_rates.append(

            log["learning_rate"]
        )

        lr_epochs.append(

            log["epoch"]
        )

if len(learning_rates) > 0:

    plt.figure(figsize=(10,6))

    plt.plot(

        lr_epochs,
        learning_rates,

        marker='o',

        linewidth=2
    )

    plt.xlabel("Epoch")

    plt.ylabel("Learning Rate")

    plt.title("Learning Rate Scheduler Curve")

    plt.grid(True)

    plt.show()

else:

    print("\nLearning rate logs not available.")

# ============================================================
# SAMPLE PREDICTIONS
# ============================================================

print("\n" + "="*100)
print("SAMPLE MODEL PREDICTIONS")
print("="*100)

sample_count = 5

sample_indices = np.random.choice(

    len(test_df),

    sample_count,

    replace=False
)

for idx_num, idx in enumerate(sample_indices):

    row = test_df.iloc[idx]

    text = row["page_text"]

    true_label = reverse_mapping[

        row["label"]
    ]

    pred_label = reverse_mapping[

        preds[idx]
    ]

    confidence = np.max(

        probabilities[idx]
    )

    print("\n" + "-"*100)

    print(f"SAMPLE {idx_num+1}")

    print("-"*100)

    print(f"\nTRUE LABEL      : {true_label}")

    print(f"PREDICTED LABEL : {pred_label}")

    print(f"CONFIDENCE      : {confidence:.4f}")

    print("\nTEXT SNIPPET:\n")

    print(text[:800])

# ============================================================
# CORRECT vs INCORRECT PREDICTIONS
# ============================================================

print("\n" + "="*100)
print("CORRECT vs INCORRECT PREDICTIONS")
print("="*100)

correct_predictions = np.sum(

    preds == labels
)

incorrect_predictions = np.sum(

    preds != labels
)

plt.figure(figsize=(7,5))

plt.bar(

    ["Correct", "Incorrect"],

    [correct_predictions, incorrect_predictions]
)

plt.title("Correct vs Incorrect Predictions")

plt.ylabel("Count")

plt.grid(axis='y')

plt.show()

# ============================================================
# CLASS-WISE F1 SCORES
# ============================================================

print("\n" + "="*100)
print("CLASS-WISE F1 SCORES")
print("="*100)

from sklearn.metrics import f1_score

class_f1_scores = f1_score(

    labels,
    preds,

    average=None
)

class_names = [

    reverse_mapping[i]

    for i in sorted(reverse_mapping.keys())
]

for class_name, score in zip(

    class_names,
    class_f1_scores
):

    print(f"{class_name:15} : {score:.4f}")

plt.figure(figsize=(10,6))

plt.bar(

    class_names,
    class_f1_scores
)

plt.title("Class-Wise F1 Scores")

plt.ylabel("F1 Score")

plt.ylim(0,1)

plt.xticks(rotation=20)

plt.grid(axis='y')

plt.show()

# ============================================================
# FINAL EXTRA ANALYSIS SUMMARY
# ============================================================

print("\n" + "="*100)
print("EXTRA ANALYSIS COMPLETED")
print("="*100)

print("""

ADDITIONAL RESEARCH COMPONENTS GENERATED

✓ Model Parameter Summary
✓ Learning Rate Scheduler Curve
✓ Real Sample Predictions
✓ Confidence Analysis
✓ Correct vs Incorrect Prediction Plot
✓ Class-Wise F1 Scores

YOUR STAGE 1 PIPELINE IS NOW RESEARCH-GRADE.

READY FOR:

✓ Stage 2 XAI
✓ SHAP
✓ LIME
✓ Attention Visualization
✓ Comparative Transformer Analysis
✓ Dissertation Reporting

""")

In [ ]:
# ============================================================
# SAMPLE PREDICTIONS
# ============================================================

print("\n" + "="*100)
print("DETAILED SAMPLE MODEL PREDICTIONS")
print("="*100)

sample_count = 30

sample_indices = np.random.choice(

    len(test_df),

    sample_count,

    replace=False
)

for idx_num, idx in enumerate(sample_indices):

    row = test_df.iloc[idx]

    text = row["page_text"]

    true_label = reverse_mapping[

        row["label"]
    ]

    pred_label = reverse_mapping[

        preds[idx]
    ]

    confidence = np.max(

        probabilities[idx]
    )

    correctness = (

        "CORRECT"

        if true_label == pred_label

        else "WRONG"
    )

    print("\n" + "="*100)

    print(f"SAMPLE {idx_num+1}")

    print("="*100)

    print(f"\nTRUE LABEL      : {true_label}")

    print(f"PREDICTED LABEL : {pred_label}")

    print(f"CONFIDENCE      : {confidence:.4f}")

    print(f"STATUS          : {correctness}")

    print("\nTEXT SNIPPET:\n")

    print(text[:1200])

    print("\n")

In [ ]:
# ============================================================
# SAVE TRAINED ROBERTA MODEL
# ============================================================

import os

SAVE_PATH = "./final_roberta_bias_model"

# ============================================================
# CREATE DIRECTORY
# ============================================================

os.makedirs(

    SAVE_PATH,

    exist_ok=True
)

# ============================================================
# SAVE MODEL
# ============================================================

print("="*100)
print("SAVING TRAINED ROBERTA MODEL")
print("="*100)

trainer.save_model(

    SAVE_PATH
)

# ============================================================
# SAVE TOKENIZER
# ============================================================

tokenizer.save_pretrained(

    SAVE_PATH
)

# ============================================================
# VERIFY FILES
# ============================================================

print("\nSaved Files:\n")

print(

    os.listdir(SAVE_PATH)
)

# ============================================================
# SUCCESS
# ============================================================

print("\n" + "="*100)
print("MODEL SAVED SUCCESSFULLY")
print("="*100)

print(f"\nSaved Location:\n{SAVE_PATH}")

# **XAI**

**LOAD MODEL + SHAP/XAI SETUP**

**MASS PREDICTIONS + DETAILED XAI**

In [ ]:
# ============================================================
# CELL 1
# LARGE SCALE PREDICTION + SHAP EXPLANATIONS
# ============================================================

# ============================================================
# SAMPLE SELECTION
# ============================================================

print("="*100)
print("GENERATING MULTIPLE XAI EXPLANATIONS")
print("="*100)

sample_count = 12

sample_df = test_df.sample(

    sample_count,

    random_state=42
).reset_index(drop=True)

# ============================================================
# PREDICTIONS + EXPLANATIONS
# ============================================================

for idx in range(sample_count):

    text = sample_df.loc[idx, "page_text"]

    true_label = label_mapping[

        sample_df.loc[idx, "label"]
    ]

    # ========================================================
    # MODEL PREDICTION
    # ========================================================

    outputs = classifier(text[:1500])

    outputs = sorted(

        outputs[0],

        key=lambda x: x["score"],

        reverse=True
    )

    pred_label = outputs[0]["label"]

    confidence = outputs[0]["score"]

    # ========================================================
    # DISPLAY INFO
    # ========================================================

    print("\n" + "="*100)

    print(f"SAMPLE {idx+1}")

    print("="*100)

    print(f"\nTRUE LABEL      : {true_label}")

    print(f"PREDICTED LABEL : {pred_label}")

    print(f"CONFIDENCE      : {confidence:.4f}")

    print("\nTEXT SNIPPET:\n")

    print(text[:1200])

    # ========================================================
    # SHAP VALUES
    # ========================================================

    print("\nGenerating SHAP Explanation...\n")

    shap_values = explainer(

        [text[:1200]]
    )

    shap.plots.text(

        shap_values[0]
    )

print("\n" + "="*100)
print("CELL 2 COMPLETED SUCCESSFULLY")
print("="*100)